This is a reimplementation of reverse mode automatic differentiation from before, but supports vectors this time around.

I don't want to reimplement matrix/vector operations, so I'll be borrowing from `numpy`

In [ ]:
import numpy as np

In [113]:
class RValue:
    
    _prev: list[tuple['RValue', np.ndarray]]
    count = 0
    
    def __init__(self, val) -> None:
        self._prev = []
        self.val = np.array(val)
        self.grad: np.ndarray | None = None
        RValue.count += 1
        self.name = f'u{RValue.count}'
        
    def _broadcast(self, other):
        self = self if self.val.shape else RValue(self.val * np.ones_like(other.val))
        other = other if other.val.shape else RValue(other.val * np.ones_like(self.val))
        return self, other
    
    def backward(self):
        ordered_nodes: list[RValue] = []
        visited = set()
        
        def build_graph(node: RValue):
            if node in visited: return
            node.grad = None
            visited.add(node)
            for parent, _ in node._prev:
                build_graph(parent)
            ordered_nodes.append(node)
        build_graph(self)
        ordered_nodes.reverse()
        
        if self.val.shape:
            self.grad = np.diag(np.ones_like(self.val))
        else:
            self.grad = np.array(1)
        for node in ordered_nodes:
            print()
            print(node)
            for parent, parent_grad in node._prev:
                print('>', parent, parent_grad)
                update = node.grad @ parent_grad
                print('>', update)
                if parent.grad is None:
                    parent.grad = update
                else:
                    parent.grad += update
    
    def __add__(self, other) -> 'RValue':
        if not isinstance(other, type(self)): other = RValue(other)
        out = RValue(self.val + other.val)
        _self, _other = self._broadcast(other)
        out._prev = [
            (self, np.diag(np.ones_like(_self.val))),
            (other, np.diag(np.ones_like(_other.val))),
        ]
        return out
    
    def __radd__(self, other) -> 'RValue':
        return self + other
    
    def __sub__(self, other) -> 'RValue':
        if not isinstance(other, type(self)): other = RValue(other)
        out = RValue(self.val - other.val)
        _self, _other = self._broadcast(other)
        out._prev = [
            (self, np.diag(np.ones_like(_self.val))),
            (other, np.diag(-np.ones_like(_other.val))),
        ]
        return out
    
    def __rsub__(self, other) -> 'RValue':
        if not isinstance(other, type(self)): other = RValue(other)
        return other - self
    
    def __mul__(self, other) -> 'RValue':
        if not isinstance(other, type(self)): other = RValue(other)
        out = RValue(self.val * other.val)
        _self, _other = self._broadcast(other)
        out._prev = [(self, np.diag(_other.val)), (other, np.diag(_self.val))]
        return out
    
    def __rmul__(self, other) -> 'RValue':
        return self * other
    
    def __truediv__(self, other) -> 'RValue':
        if not isinstance(other, type(self)): other = RValue(other)
        out = RValue(self.val / other.val)
        _self, _other = self._broadcast(other)
        out._prev = [(self, np.diag(1 / _other.val)), (other, np.diag(-_self.val / (_other.val ** 2)))]
        return out
    
    def __rtruediv__(self, other) -> 'RValue':
        if not isinstance(other, type(self)): other = RValue(other)
        return other / self
    
    def __pow__(self, other) -> 'RValue':
        if not isinstance(other, type(self)): other = RValue(other)
        out = RValue(self.val ** other.val)
        _self, _other = self._broadcast(other)
        out._prev = [(self, _other.val * _self.val ** (_other.val - 1)),
                     (other, _self.val ** _other.val * np.log(_self.val))]
        return out
    
    def __rpow__(self, other) -> 'RValue':
        if not isinstance(other, type(self)): other = RValue(other)
        return other ** self
    
    def __repr__(self) -> str:
        return f'{self.name}=RValue({self.val}, {self.grad})'

def Output(out: RValue) -> RValue:
    out.backward()
    return out

In [92]:
x = RValue([2, 3, 5]); x.name = 'x'
w = RValue([1, 2, 0]); w.name = 'w'
b = RValue(5); b.name = 'b'

y = w*x+b; y.name = 'y'
Output(y)


y=RValue([ 7 11  5], [[1 0 0]
 [0 1 0]
 [0 0 1]])
> u10=RValue([2 6 0], None) [[1 0 0]
 [0 1 0]
 [0 0 1]]
> [[1 0 0]
 [0 1 0]
 [0 0 1]]
> b=RValue(5, None) [[1 0 0]
 [0 1 0]
 [0 0 1]]
> [[1 0 0]
 [0 1 0]
 [0 0 1]]

b=RValue(5, [[1 0 0]
 [0 1 0]
 [0 0 1]])

u10=RValue([2 6 0], [[1 0 0]
 [0 1 0]
 [0 0 1]])
> w=RValue([1 2 0], None) [[2 0 0]
 [0 3 0]
 [0 0 5]]
> [[2 0 0]
 [0 3 0]
 [0 0 5]]
> x=RValue([2 3 5], None) [[1 0 0]
 [0 2 0]
 [0 0 0]]
> [[1 0 0]
 [0 2 0]
 [0 0 0]]

x=RValue([2 3 5], [[1 0 0]
 [0 2 0]
 [0 0 0]])

w=RValue([1 2 0], [[2 0 0]
 [0 3 0]
 [0 0 5]])


y=RValue([ 7 11  5], [[1 0 0]
 [0 1 0]
 [0 0 1]])

In [114]:
from typing import Callable

class Function:
    def __init__(self, _forward_backward: Callable[[RValue], tuple[float, float]]) -> None:
        self._forward_backward = _forward_backward
    
    def __call__(self, x: RValue) -> RValue:
        self.x = x
        f, dx = self._forward_backward(x)
        
        new_node = RValue(f)
        new_node._prev = [(x, dx)]
        return new_node


exp = Function(lambda x: (np.exp(x.val), np.exp(x.val)))
sin = Function(lambda x: (np.sin(x.val), np.cos(x.val)))
cos = Function(lambda x: (np.cos(x.val), -np.sin(x.val)))
log = Function(lambda x: (np.log(x.val), 1/x.val))
sum = Function(lambda x: (np.sum(x.val), np.ones_like(x.val).reshape((1,-1))))

In [115]:
x = RValue([2, 3, 5]); x.name = 'x'
w = RValue([1, 2, 0]); w.name = 'w'
b = RValue(5); b.name = 'b'

z = w*x+b; z.name = 'z'
y = sum(z); y.name = 'y'

Output(y)


y=RValue(23, 1)
> z=RValue([ 7 11  5], None) [[1 1 1]]


ValueError: matmul: Input operand 0 does not have enough dimensions (has 0, gufunc core with signature (n?,k),(k,m?)->(n?,m?) requires 1)

In [107]:
x.val.size

3